# Word2Vec ile duygu analizi

### Alıştırma hedefleri:
- Word2Vec ile kelimeleri vektörlere dönüştürmek
- Word2vec tarafından verilen kelime temsilini RNN'ye beslemek için kullanmak

<hr>

▶️ Bu hücreyi çalıştırın ve kullandığınız 📚 [Gensim - Word2Vec](https://radimrehurek.com/gensim/auto_examples/index.html) sürümünün ≥ 4.0 olduğundan emin olun!

In [1]:
!pip freeze | grep gensim

gensim==4.3.3


# Veri


❓ **Soru** ❓ Öncelikle verileri yükleyelim. Fonksiyonda neler olduğunu anlamanıza gerek yok, burada önemi yok.

⚠️ **Uyarı** ⚠️ `load_data` fonksiyonunda `percentage_of_sentences` argümanı vardır. Bilgisayarınıza bağlı olarak, çok fazla cümle bilgisayarınızı yavaşlatabilir veya hatta dondurabilir - RAM'iniz taşabilir. Bu nedenle, **cümlelerin %10'uyla başlamalı** ve bilgisayarınızın bunu kaldırabildiğini kontrol etmelisiniz. Aksi takdirde, daha düşük bir sayı ile yeniden çalıştırın. 

⚠️ **DISCLAIMER** ⚠️ **_En büyüğü kimde_ (_who has the biggest_)(RAM) oyununu oynamaya gerek yok!** Buradaki amaç, modellerinizi hızlı bir şekilde çalıştırarak prototip oluşturmaktır. Gerçek hayatta bile, hızlı bir şekilde döngü ve hata ayıklama yapmak için verilerinizin bir alt kümesiyle başlamanız önerilir. Bu nedenle, yalnızca en iyi doğruluğu elde etmek istiyorsanız sayıyı artırın.

In [ ]:
import numpy as np

In [2]:
####################################################
### Verileri yüklemek için bu hücreyi çalıştırın ###
####################################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Tüm verilerin yalnızca belirli bir yüzdesini alın
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

2026-01-30 00:53:49.912891: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 00:53:49.919599: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 00:53:49.979238: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 00:53:50.044149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-30 00:53:50.104041: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

Önceki alıştırmada, Word2vec temsilini eğittiniz ve bu Şekil'in ilk adımında gösterildiği gibi, tüm eğitim cümlelerinizi bir RNN'ye beslemek için dönüştürdünüz: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word2vec_representation.png" alt="Word2Vec with RNN" width="400px" />



❓ **Soru** ❓ Burada, önceki alıştırmada yaptığınızın aynısını tekrar yapalım. İlk olarak, eğitim cümleniz üzerinde bir word2vec modeli (istediğiniz argümanlarla) eğitin. Bunu `word2vec` değişkenine kaydedin.

In [3]:
from gensim.models import Word2Vec

word2vec = Word2Vec(sentences=X_train, vector_size=53, min_count=13, window=7)

Önceki alıştırmadaki işlevleri yeniden kullanarak, eğitim ve test verilerinizi RNN'ye girebileceğiniz bir biçime dönüştürelim.

❓ **Soru** ❓ Neler olduğunu anladığınızdan emin olmak için aşağıdaki işlevi okuyun ve çalıştırın.

In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec.wv:
            embedded_sentence.append(word2vec.wv[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün(embed edin)
X_train_embed = embedding(word2vec, X_train)
X_test_embed = embedding(word2vec, X_test)


# Eğitim ve test gömülü cümleleri doldurun
X_train_pad = pad_sequences(X_train_embed, dtype='float32', padding='post', maxlen=200)
X_test_pad = pad_sequences(X_test_embed, dtype='float32', padding='post', maxlen=200)

☝️ Çalıştığından emin olmak için, `X_train_pad` ve `X_test_pad` için aşağıdakileri kontrol edelim:

- bunlar numpy dizileridir
- bunlar 3 boyutludur
- son boyut, word2vec gömme alanınızın boyutundadır (bunu `word2vec.wv.vector_size` ile elde edebilirsiniz
- ilk boyut, `X_train` ve `X_test` boyutundadır

✅ **İyi Uygulama** ✅ Bu tür testler oldukça önemlidir! Sadece bu alıştırmada değil, gerçek hayattaki uygulamalarda da. Hataları çok geç fark etmeyi ve bunların tüm not defterine yayılmasını önler.

In [5]:
# BENİ TEST ET
for X in [X_train_pad, X_test_pad]:
    assert type(X) == np.ndarray
    assert X.shape[-1] == word2vec.wv.vector_size


assert X_train_pad.shape[0] == len(X_train)
assert X_test_pad.shape[0] == len(X_test)

# Temel model

Kendi modelinizi test etmek için çok basit bir modele sahip olmak her zaman iyidir - çok basit bir algoritmadan daha iyi bir şey yaptığınızdan emin olmak için.

❓ **Soru** ❓ Temel doğruluk oranınız nedir? Bu durumda, temeliniz `y_train` içinde en çok bulunan etiketi tahmin etmek olabilir (tabii ki, veri kümesi dengeli ise, temel doğruluk oranı 1/n'dir; burada n, sınıfların sayısıdır - burada 2'dir).

In [6]:
from sklearn.metrics import accuracy_score

unique, counts = np.unique(y_train, return_counts=True)
counts = dict(zip(unique, counts))

print('Number of labels in train set', counts)

y_pred = 0 if counts[0] > counts[1] else 1

print('Baseline accuracy: ', accuracy_score(y_test, [y_pred]*len(y_test)))

Number of labels in train set {0: 1265, 1: 1235}
Baseline accuracy:  0.492


# Model

❓ **Soru** ❓ Aşağıdaki katmanlara sahip bir RNN yazın:
- bir `Masking` katmanı
- 20 birim ve `tanh` aktivasyon fonksiyonuna sahip bir `LSTM`
- 10 birimlik bir `Dense`
- görevinize bağlı bir çıktı katmanı

Ardından, modelinizi derleyin (en azından başlangıçta optimizer olarak `rmsprop` kullanmanızı öneririz).

In [7]:
from tensorflow.keras import Sequential
from tensorflow.keras import layers

def init_model():
    model = Sequential()
    model.add(layers.Masking())
    model.add(layers.LSTM(20, activation="tanh"))
    model.add(layers.Dense(15, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])

    return model

model = init_model()

❓ **Soru** ❓ Modeli gömülü ve doldurulmuş verilerinize uyarlayın - erken durdurma kriterini unutmayın.

❗ **Not** ❗ Doğruluğunuz büyük ölçüde eğitim kümenize bağlı olacaktır. Burada, performansınızın temel modelin üzerinde olduğundan emin olun (bu, başlangıçtaki IMDB verilerinin yalnızca %20'sini yüklemiş olsanız bile geçerli olmalıdır).

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(patience=4, restore_best_weights=True)

model.fit(X_train_pad,y_train,batch_size=32,epochs=100,validation_split=0.3,callbacks=[es])

Epoch 1/100
 1/55 ━━━━━━━━━━━━━━━━━━━━ 1:07 1s/step - accuracy: 0.5000 - loss: 0.6942

2026-01-30 01:13:57.376284: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5234 - loss: 0.6930 - val_accuracy: 0.5733 - val_loss: 0.6833
Epoch 2/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5777 - loss: 0.6803 - val_accuracy: 0.5933 - val_loss: 0.6700
Epoch 3/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.5920 - loss: 0.6696 - val_accuracy: 0.5893 - val_loss: 0.6647
Epoch 4/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.6063 - loss: 0.6603 - val_accuracy: 0.6067 - val_loss: 0.6617
Epoch 5/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.6223 - loss: 0.6467 - val_accuracy: 0.6107 - val_loss: 0.6519
Epoch 6/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.6360 - loss: 0.6317 - val_accuracy: 0.6200 - val_loss: 0.6522
Epoch 7/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.6446 - loss: 0.6225 - val_accuracy: 0.6493 - val_loss: 0.6279
Epoch 8/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.6703 - loss: 0.6080 - val_accuracy: 0.6600 - val_l

❓ **Soru** ❓ Test setinde modelinizi değerlendirin

In [10]:
res = model.evaluate(X_test_pad, y_test, verbose=0)

print(f'The accuracy evaluated on the test set is of {res[1]*100:.3f}%')

The accuracy evaluated on the test set is of 66.440%


# Eğitimli Word2Vec - Transfer Öğrenimi

Doğruluğunuz, temel modelin üzerinde olsa da, oldukça düşük olabilir. Veri temizleme ve gömme kalitesini iyileştirme gibi bunu iyileştirmek için birçok seçenek vardır.

Burada veri temizleme stratejilerine girmeyeceğiz. Gömme kalitemizi iyileştirmeye çalışalım. Ancak, daha büyük bir metin kümesini yüklemek yerine, neden başkalarının öğrendiği gömme modelinden yararlanmayalım? Çünkü gömme modelinin kalitesi, yani kelimelerin yakınlığı, farklı görevlerden elde edilebilir. Transfer öğrenme tam olarak budur.

❓ **Soru** ❓ Bu sayede word2vec'te bulunan tüm farklı modelleri listeleyin: 

In [11]:
import gensim.downloader as api
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


ℹ️ Modellerin listesini ve boyutlarını [`gensim-data` deposunda](https://github.com/RaRe-Technologies/gensim-data#models) de bulabilirsiniz.

❓ **Soru** ❓ Önceden eğitilmiş word2vec gömme alanlarından birini yükleyin.

Bunu `api.load(seçtiğiniz model)` ile yapabilir ve `word2vec_transfer` içinde saklayabilirsiniz.

<details>
    <summary>💡 İpucu</summary>
    
`glove-wiki-gigaword-50` modeli, daha küçük olması (65 MB) nedeniyle başlangıç için iyi bir seçenektir.

</details>

In [12]:
word2vec_transfer = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


❓ **Soru** ❓ Kelime dağarcığının boyutunu ve aynı zamanda gömme alanının boyutunu kontrol edin.

In [15]:
print(len(word2vec_transfer.key_to_index))
print(len(word2vec_transfer['genius']))

400000
50


❓ İlk soruda yaptığımız gibi, `X_train` ve `X_test`'i gömelim! (`embed_sentence_with_TF` işlevinde küçük bir fark var, ancak bu konuya girmeyeceğiz.)

In [16]:
# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence_with_TF(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec:
            embedded_sentence.append(word2vec[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence_with_TF(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün (embed edin)
X_train_embed_2 = embedding(word2vec_transfer, X_train)
X_test_embed_2 = embedding(word2vec_transfer, X_test)

❓ **Soru** ❓  Sonuçlarınızı doldurmayı ve bunları `X_train_pad_2` ve `X_test_pad_2` içinde saklamayı unutmayın.

In [19]:
X_train_pad_2 = pad_sequences(X_train_embed_2, dtype="float32", padding="post", maxlen=200)

X_test_pad_2 = pad_sequences(X_test_embed_2, dtype="float32", padding="post", maxlen=200)

❓ **Soru** ❓ Modeli yeniden başlatın ve yeni gömülü (ve doldurulmuş(padded)) verilerinize uyarlayın!  Test setinizde değerlendirin ve önceki doğruluğunuzla karşılaştırın.

❗ **Not** ❗ Buradaki eğitim biraz zaman alabilir. Sadece 10 dönem hesaplayabilir (bu **iyi** bir uygulama değildir, sadece çok uzun süre beklememek içindir) ve eğitim sürerken bir sonraki alıştırmaya geçebilir veya bir mola verebilirsiniz, muhtemelen bunu hak ettiniz ;)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(patience=4, restore_best_weights=True)

model = init_model()

model.fit(X_train_pad_2, y_train, batch_size=32, epochs=15, validation_split=0.3, callbacks=[es])

Epoch 1/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5109 - loss: 0.6922 - val_accuracy: 0.5760 - val_loss: 0.6820
Epoch 2/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.6120 - loss: 0.6704 - val_accuracy: 0.6480 - val_loss: 0.6499
Epoch 3/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.6514 - loss: 0.6354 - val_accuracy: 0.6547 - val_loss: 0.6327
Epoch 4/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.6846 - loss: 0.6089 - val_accuracy: 0.7000 - val_loss: 0.5871
Epoch 5/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - accuracy: 0.6960 - loss: 0.5926 - val_accuracy: 0.6840 - val_loss: 0.6194
Epoch 6/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7006 - loss: 0.5799 - val_accuracy: 0.6707 - val_loss: 0.6157
Epoch 7/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.7126 - loss: 0.5660 - val_accuracy: 0.7280 - val_loss: 0.5572
Epoch 8/15
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.7171 - loss: 0.5563 - val_accuracy: 0.7373 - v

In [22]:
res = model.evaluate(X_test_pad_2, y_test, verbose=0)

print(f'The accuracy evaluated on the test set is of {res[1]*100:.3f}%')

The accuracy evaluated on the test set is of 74.120%


Yeni word2vec'iniz büyük bir metin kümesinde eğitildiğinden, çok sayıda kelimeyi temsil eder! Küçük veri kümenize kıyasla çok daha fazladır, özellikle de eğitim kümesinde belirli bir sayıdan fazla bulunmayan kelimeleri elediğiniz için. Bu nedenle, eğitim ve test kümenizde çok daha fazla gömülü kelime vardır, bu da her yinelemeyi öncekinden daha uzun hale getirir.